# Week 25, Day 6: Project report and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w025/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.load(dataset("churchland2012_reach.npz"))
rates, times = data["rates"].astype(float), data["times"]
C, T, N = rates.shape
dt = 0.01

def preprocess(soft):
    Xs = rates / (rates.max(axis=(0, 1)) - rates.min(axis=(0, 1)) + soft + 1e-9)
    return Xs - Xs.mean(axis=0, keepdims=True)

def rotation_r2(A, k=6):
    D = A.reshape(-1, A.shape[-1]); mu = D.mean(0)
    Vt = np.linalg.svd(D - mu, full_matrices=False)[2]
    Z = (A - mu) @ Vt[:k].T
    Zm, dZ = Z[:, :-1].reshape(-1, k), (np.diff(Z, axis=1) / dt).reshape(-1, k)
    basis = []
    for i in range(k):
        for j in range(i + 1, k):
            E = np.zeros((k, k)); E[i, j], E[j, i] = 1, -1; basis.append(E)
    G = np.stack([(Zm @ E.T).ravel() for E in basis], axis=1)
    h = np.linalg.lstsq(G, dZ.ravel(), rcond=None)[0]
    return 1 - np.sum((dZ.ravel() - G @ h) ** 2) / np.sum((dZ - dZ.mean(0)) ** 2)
print("ready")

### Exercise: Does the result depend on my choices?

Fill `grid` with the rotational R² for every combination of soft-normalisation constant (0, 5, 20 spikes/s), number of components (4, 6, 8) and window ((−50, 150) or (0, 200) ms), keyed by the tuple (soft, k, window).

In [ ]:
grid = {}
for soft in (0, 5, 20):
    X = preprocess(soft)
    for k in (4, 6, 8):
        for window in ((-50, 150), (0, 200)):
            mask = (times >= window[0]) & (times <= window[1])
            grid[(soft, k, window)] = ...
for key, v in grid.items():
    print(key, round(float(v), 3))
print(f"range {min(grid.values()):.2f} to {max(grid.values()):.2f}")

In [ ]:
import numpy as np
def test_grid():
    "grid holds 18 values, one per combination, keyed by (soft, k, window)"
    assert len(grid) == 18 and all(len(key) == 3 for key in grid)

def test_robust():
    "every combination gives a rotational R squared above 0.2, at least ten times the largest null (0.02)"
    assert min(grid.values()) > 0.2

test_grid()
test_robust()
print("All checks passed")

### Exercise: The report's numbers

Complete `project_summary()` so that it computes, from the raw rates with soft constant 5, six components and the window −50 to 150 ms: the numbers of neurons and conditions, the variance in the top six components, the participation ratio, the R² of the free and rotational fits, and the top rotation frequency in Hz. Store its result in `summary`.

In [ ]:
def project_summary():
    X = preprocess(5)
    A = X[:, (times >= -50) & (times <= 150)]
    D = A.reshape(-1, N); mu = D.mean(0)
    _, S, Vt = np.linalg.svd(D - mu, full_matrices=False)
    lam = S ** 2
    Z = (A - mu) @ Vt[:6].T
    Zm, dZ = Z[:, :-1].reshape(-1, 6), (np.diff(Z, axis=1) / dt).reshape(-1, 6)
    M_full = np.linalg.lstsq(Zm, dZ, rcond=None)[0].T
    r2_full = 1 - np.sum((dZ - Zm @ M_full.T) ** 2) / np.sum((dZ - dZ.mean(0)) ** 2)
    basis = []
    for i in range(6):
        for j in range(i + 1, 6):
            E = np.zeros((6, 6)); E[i, j], E[j, i] = 1, -1; basis.append(E)
    G = np.stack([(Zm @ E.T).ravel() for E in basis], axis=1)
    M_skew = sum(c * E for c, E in zip(np.linalg.lstsq(G, dZ.ravel(), rcond=None)[0], basis))
    return {
        "n_neurons": N, "n_conditions": C,
        "var_top6": ...,
        "participation_ratio": ...,
        "r2_full": r2_full,
        "r2_rotation": rotation_r2(A),
        "rotation_hz": ...,
    }

summary = project_summary()
for k, v in summary.items():
    print(f"{k:22s} {v:.3f}" if isinstance(v, float) else f"{k:22s} {v}")

In [ ]:
import numpy as np
def test_keys():
    "summary returns all the report's quantities"
    need = {"n_neurons", "n_conditions", "var_top6", "participation_ratio", "r2_full", "r2_rotation", "rotation_hz"}
    assert need <= set(summary.keys())

def test_values():
    "the values agree with the week's results"
    assert summary["n_neurons"] == 218 and summary["n_conditions"] == 108
    assert abs(summary["var_top6"] - 0.611) < 0.005 and abs(summary["r2_rotation"] - 0.489) < 0.005
    assert 10 < summary["participation_ratio"] < 11.5 and abs(summary["rotation_hz"] - 1.27) < 0.02 and summary["r2_full"] > summary["r2_rotation"]

test_keys()
test_values()
print("All checks passed")